In [60]:
import pandas as pd
import numpy as np
import string
import nltk
import warnings
import joblib

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
from string import punctuation

from sklearn.preprocessing import LabelEncoder, MaxAbsScaler,StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

from xgboost import XGBClassifier
from scipy.stats import loguniform

warnings.filterwarnings("ignore")





In [3]:
df=pd.read_csv("IMDB Dataset.csv")

In [4]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [5]:
label=LabelEncoder()
df['sentiment']=label.fit_transform(df['sentiment'])

In [6]:
df.isnull().sum()

,0
review,0
sentiment,0


In [34]:
df['review'][0]

'one reviewer mentioned watching 1 oz episode hooked right exactly happened br br first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use br br called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home many aryan muslim gangsta latino christian italian irish scuffle death stare dodgy dealing shady agreement never far br br would say main appeal show due fact go show would dare forget pretty picture painted mainstream audience forget charm forget romance oz mess around first episode ever saw struck nasty surreal could say ready watched developed taste oz got accustomed high level graphic violence violence injustice crooked guard sold nickel inmate kill order get away well mannered middle class inmate turned prison bitch due lack street skill prison ex

In [8]:
!pip install nltk

In [10]:
def transform(text):
    text=text.lower()
    text=nltk.word_tokenize(text)
    text=[word for word in text if word.isalnum()]
    text=[word for word in text if word not in stopwords.words('english')]
    text=[WordNetLemmatizer().lemmatize(word) for word in text]
    return " ".join(text)

In [11]:
transform('''One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fac..''')

'one reviewer mentioned watching 1 oz episode hooked right exactly happened br br first thing struck oz brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use br br called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inwards privacy high agenda em city home many aryan muslim gangsta latino christian italian irish scuffle death stare dodgy dealing shady agreement never far br br would say main appeal show due fac'

In [12]:
df['review']=df['review'].apply(transform)

In [14]:
df.to_csv("clean_data.csv", index=False)

In [20]:


# ========== 1. VECTORIZATION ==========
tf = TfidfVectorizer(max_features=3000)
x = tf.fit_transform(df['review']).toarray()
y = df['sentiment'].values


In [21]:


sc = StandardScaler()
x = sc.fit_transform(x)

# ========== 3. TRAIN-TEST SPLIT ==========
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)


In [33]:


# ========== 1. VECTORIZATION ==========
tf = TfidfVectorizer(max_features=3000)
x = tf.fit_transform(df['review'])        # ← Keep as sparse matrix (don't use .toarray())
y = df['sentiment'].values

# ========== 2. SCALING ==========
sc = MaxAbsScaler()                       # ← FIXED: MaxAbsScaler works with sparse data
x = sc.fit_transform(x)                   # Sparse matrix preserved

# ========== 3. TRAIN-TEST SPLIT ==========
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# ========== 4. TRAIN XGBOOST ==========
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

xgb.fit(x_train, y_train)

# ========== 5. EVALUATE ==========
y_train_pred = xgb.predict(x_train)
y_test_pred = xgb.predict(x_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_test_pred))
print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))



# ========== 6. PREDICT NEW MOVIE REVIEW ==========
def predict_review(review_text):
    # Transform the new review
    review_tfidf = tf.transform([review_text])    # Returns sparse matrix
    review_scaled = sc.transform(review_tfidf)    # MaxAbsScaler handles sparse

    # Predict
    prediction = xgb.predict(review_scaled)[0]
    probability = xgb.predict_proba(review_scaled)[0]

    sentiment = "Positive" if prediction == 1 else "Negative"
    confidence = probability[prediction]

    return sentiment, confidence

# ========== 7. TEST PREDICTIONS ==========
print("\n" + "=" * 50)
print("🎬 MOVIE REVIEW PREDICTIONS")
print("=" * 50)

reviews = [
    "This movie was absolutely amazing! Best film I've ever seen.",
    "Terrible waste of time. Boring plot and bad acting.",
    "The cinematography was stunning but the story was weak.",
    "A masterpiece! Brilliant performances and gripping storyline.",
    "I fell asleep halfway through. Completely dull and predictable."
]

for review in reviews:
    sentiment, confidence = predict_review(review)
    emoji = "😊" if sentiment == "Positive" else "😞"
    print(f"\n{emoji} Review: '{review}'")
    print(f"   Sentiment: {sentiment}")
    print(f"   Confidence: {confidence:.2%}")

Train Accuracy: 0.9287
Test Accuracy:  0.8548

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.85      5000
           1       0.84      0.87      0.86      5000

    accuracy                           0.85     10000
   macro avg       0.86      0.85      0.85     10000
weighted avg       0.86      0.85      0.85     10000


📊 Confusion Matrix:
[[4177  823]
 [ 629 4371]]

✅ Model saved!

🎬 MOVIE REVIEW PREDICTIONS

😊 Review: 'This movie was absolutely amazing! Best film I've ever seen.'
   Sentiment: Positive
   Confidence: 93.70%

😞 Review: 'Terrible waste of time. Boring plot and bad acting.'
   Sentiment: Negative
   Confidence: 98.86%

😊 Review: 'The cinematography was stunning but the story was weak.'
   Sentiment: Positive
   Confidence: 55.38%

😊 Review: 'A masterpiece! Brilliant performances and gripping storyline.'
   Sentiment: Positive
   Confidence: 78.12%

😞 Review: 'I fell asleep halfway through. Com

In [61]:


reviews = ['''
not intrested
'''
]

for review in reviews:
    sentiment, confidence = predict_review(review)
    emoji = "😊" if sentiment == "Positive" else "😞"
    print(f"\n{emoji} Review: '{review}'")
    print(f"   Sentiment: {sentiment}")
    print(f"   Confidence: {confidence:.2%}")


😊 Review: '
not intrested
'
   Sentiment: Positive
   Confidence: 50.37%


In [43]:
from sklearn.svm import LinearSVC

# LinearSVC is faster than SVC for high-dimensional text data
model = LinearSVC(C=1.0, max_iter=2000, random_state=42)
model.fit(x_train, y_train)

# For probability output (optional)
from sklearn.calibration import CalibratedClassifierCV
model = CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000), cv=3)
model.fit(x_train, y_train)
print(accuracy_score(y_test, model.predict(x_test)))
print(confusion_matrix(y_test, model.predict(x_test)))


0.8782
[[4356  644]
 [ 574 4426]]


In [44]:
from sklearn.naive_bayes import MultinomialNB

# ⚠️ Don't use MaxAbsScaler with Naive Bayes! Use raw TF-IDF or CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

tf = CountVectorizer(max_features=3000)
x = tf.fit_transform(df['review'])  # Keep sparse, no scaling needed

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

model = MultinomialNB(alpha=1.0)
model.fit(x_train, y_train)
print(accuracy_score(y_test, model.predict(x_test)))
print(confusion_matrix(y_test, model.predict(x_test)))

0.8441
[[4206  794]
 [ 765 4235]]


In [48]:
# Base model
base_svc = LinearSVC(random_state=42, max_iter=5000)

# Parameter distribution (random sampling)
param_dist = {
    'estimator__C': loguniform(0.01, 100),        # Random log-scale values
    'estimator__loss': ['hinge', 'squared_hinge'],
    'estimator__class_weight': [None, 'balanced'],
    'cv': [3, 5]                                  # Random CV folds
}

# Calibrated wrapper for probability output
calibrated_svc = CalibratedClassifierCV(base_svc, method='sigmoid')

# RandomizedSearchCV - FAST! Only tests 10 random combinations
random_search = RandomizedSearchCV(
    calibrated_svc,
    param_distributions={
        'estimator__C': loguniform(0.01, 100),
        'estimator__loss': ['hinge', 'squared_hinge'],
        'estimator__class_weight': [None, 'balanced']
    },
    n_iter=10,              # Only 10 random combos (vs hundreds in GridSearch)
    cv=3,                   # 3-fold CV
    scoring='accuracy',
    n_jobs=-1,              # Use all CPU cores
    random_state=42,
    verbose=1
)

# Fit
random_search.fit(x_train, y_train)

# ========== 3. RESULTS ==========
print("=" * 50)
print("⚡ FAST TUNING RESULTS")
print("=" * 50)
print(f"Best Params: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f}")

best_model = random_search.best_estimator_

# ========== 4. EVALUATE ==========
y_pred = best_model.predict(x_test)

print(f"\nTest Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ========== 5. SAVE ==========
joblib.dump(best_model, 'linearsvc_tuned_model.pkl')
joblib.dump(tf, 'tfidf_vectorizer.pkl')
joblib.dump(sc, 'scaler.pkl')
print("\n✅ Model saved!")

# ========== 6. PREDICT NEW REVIEW ==========
def predict_review(text):
    review_tfidf = tf.transform([text])
    review_scaled = sc.transform(review_tfidf)
    pred = best_model.predict(review_scaled)[0]
    prob = best_model.predict_proba(review_scaled)[0]
    sentiment = "Positive" if pred == 1 else "Negative"
    return sentiment, prob[pred]

# Test
reviews = [
    "Amazing movie! Best I've ever seen.",
    "Terrible waste of time. So boring."
]

for r in reviews:
    sent, conf = predict_review(r)
    print(f"\n'{r}' → {sent} ({conf:.2%})")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
⚡ FAST TUNING RESULTS
Best Params: {'estimator__C': np.float64(0.012087541473056965), 'estimator__class_weight': 'balanced', 'estimator__loss': 'squared_hinge'}
Best CV Score: 0.8735

Test Accuracy: 0.8816

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      5000
           1       0.88      0.89      0.88      5000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000


Confusion Matrix:
[[4373  627]
 [ 557 4443]]

✅ Model saved!

'Amazing movie! Best I've ever seen.' → Positive (93.98%)

'Terrible waste of time. So boring.' → Negative (98.76%)


In [57]:
def predict_review(text):
    review_tfidf = tf.transform([text])
    review_scaled = sc.transform(review_tfidf)
    pred = best_model.predict(review_scaled)[0]
    prob = best_model.predict_proba(review_scaled)[0]
    sentiment = "Positive" if pred == 1 else "Negative"
    return sentiment, prob[pred]

# Test
reviews = ['''
 bad
'''

]

for r in reviews:
    sent, conf = predict_review(r)
    print(f"\n'{r}' → {sent} ({conf:.2%})")


'
 bad
' → Negative (61.96%)
